In [4]:
import sys
import os
import pandas as pd

print(sys.executable)
print(os.getcwd())

df = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
print(df.shape)


d:\ml\WB-hackathon\vscode\.venv\Scripts\python.exe
d:\ml\WB-hackathon\vscode
(4630000, 9)


In [5]:
df.head()

,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h
0,471,2025-07-28 00:00:00,13,23,45,85645,491354,487,88655
1,471,2025-07-28 00:30:00,14,18,1,80049,494754,489,223929
2,471,2025-07-28 01:00:00,8,8,38,82616,490659,487,168519
3,471,2025-07-28 01:30:00,10,2,7,80506,489967,494,111380
4,471,2025-07-28 02:00:00,7,5,13,85992,479743,494,246046


In [6]:
df.columns

Index(['route_id', 'timestamp', 'status_1', 'status_2', 'status_3', 'status_4',
       'status_5', 'status_6', 'target_1h'],
      dtype='str')

In [7]:
df.shape

(4630000, 9)

In [8]:
df.dtypes

route_id              int64
timestamp    datetime64[ns]
status_1              int64
status_2              int64
status_3              int64
status_4              int64
status_5              int64
status_6              int64
target_1h             int64
dtype: object

In [9]:
df.describe()

,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h
count,4.630000e+06,4630000,4.630000e+06,4.630000e+06,4.630000e+06,4.630000e+06,4.630000e+06,4.630000e+06,4.630000e+06
mean,4.995000e+02,2025-09-14 05:14:59.999999488,2.267677e+01,2.285854e+01,2.285684e+01,1.192156e+05,7.019789e+05,3.344607e+03,2.612421e+05
min,0.000000e+00,2025-07-28 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,1.390000e+03,1.851900e+04,0.000000e+00,0.000000e+00
25%,2.497500e+02,2025-08-21 02:30:00,1.300000e+01,1.300000e+01,8.000000e+00,7.496700e+04,3.897790e+05,4.600000e+01,1.389900e+05
50%,4.995000e+02,2025-09-14 05:15:00,2.100000e+01,2.100000e+01,1.900000e+01,1.099120e+05,6.188360e+05,1.620000e+02,2.357690e+05
75%,7.492500e+02,2025-10-08 08:00:00,3.000000e+01,3.000000e+01,3.300000e+01,1.523090e+05,9.155220e+05,1.243000e+03,3.551750e+05
max,9.990000e+02,2025-11-01 10:30:00,8.020000e+02,5.220000e+02,6.200000e+02,5.879380e+05,3.503114e+06,1.527390e+05,1.083134e+07
std,2.886750e+02,NaN,1.304141e+01,1.285778e+01,1.943999e+01,5.982565e+04,4.213063e+05,1.004817e+04,1.690826e+05


In [10]:
df.head(10)

,route_id,timestamp,status_1,status_2,status_3,status_4,status_5,status_6,target_1h
0,471,2025-07-28 00:00:00,13,23,45,85645,491354,487,88655
1,471,2025-07-28 00:30:00,14,18,1,80049,494754,489,223929
2,471,2025-07-28 01:00:00,8,8,38,82616,490659,487,168519
3,471,2025-07-28 01:30:00,10,2,7,80506,489967,494,111380
4,471,2025-07-28 02:00:00,7,5,13,85992,479743,494,246046
5,471,2025-07-28 02:30:00,13,13,9,84988,484448,484,279432
6,471,2025-07-28 03:00:00,16,18,2,85730,484198,469,144766
7,471,2025-07-28 03:30:00,8,16,1,78109,490481,496,0
8,471,2025-07-28 04:00:00,10,11,22,81731,484597,480,5331
9,471,2025-07-28 04:30:00,12,14,36,83616,491429,488,95108


In [11]:
df["route_id"].nunique()

1000

In [12]:
df["timestamp"].min(), df["timestamp"].max()

(Timestamp('2025-07-28 00:00:00'), Timestamp('2025-11-01 10:30:00'))

In [13]:
df["status_1"].value_counts().head()

status_1
17    149612
15    148813
20    148671
16    148141
19    147548
Name: count, dtype: int64

In [14]:
df = df.sort_values("timestamp")

split_date = df["timestamp"].quantile(0.8)

train_df = df[df["timestamp"] <= split_date]
val_df   = df[df["timestamp"] > split_date]

In [15]:
df = df.sort_values("timestamp").reset_index(drop=True)

In [16]:
features = [
    "status_1", "status_2", "status_3",
    "status_4", "status_5", "status_6"
]

target = "target_1h"

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

In [17]:
!pip install lightgbm

In [18]:
print(X_train.shape, X_val.shape)

(3704000, 6) (926000, 6)


In [19]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
pred_val = model.predict(X_val)

In [ ]:
import numpy as np

def metric(y_true, y_pred):
    wape = np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias

print(metric(y_val.values, pred_val))

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})

importance = importance.sort_values("importance", ascending=False)
importance

In [ ]:
import matplotlib.pyplot as plt

importance.sort_values("importance").plot(
    x="feature",
    y="importance",
    kind="barh"
)

plt.show()

2 модель + lag + hour + dayofweek             
        пред: ~0,45





In [ ]:
df2 = pd.read_parquet("train_solo_track.parquet")
df2 = df2.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

df2["lag_1"] = df2.groupby("route_id")["target_1h"].shift(1)
df2["hour"] = df2["timestamp"].dt.hour
df2["dayofweek"] = df2["timestamp"].dt.dayofweek

df2 = df2.dropna().reset_index(drop=True)

In [ ]:
split_date = df2["timestamp"].quantile(0.8)

train_df = df2[df2["timestamp"] <= split_date]
val_df   = df2[df2["timestamp"] > split_date]

In [ ]:
features = [
    "status_1", "status_2", "status_3",
    "status_4", "status_5", "status_6",
    "lag_1", "hour", "dayofweek"
]

target = "target_1h"

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

In [ ]:
import lightgbm as lgb
import numpy as np

model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

pred_val = model.predict(X_val)

def metric(y_true, y_pred):
    wape = np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias

score = metric(y_val.values, pred_val)
print("Validation score:", score)

3 модель  (+ 2 lags)

пред: ~0.36

In [ ]:
df3 = pd.read_parquet("train_solo_track.parquet")
df3 = df3.sort_values(["route_id", "timestamp"]).reset_index(drop=True)

df3["lag_1"] = df3.groupby("route_id")["target_1h"].shift(1)
df3["lag_2"] = df3.groupby("route_id")["target_1h"].shift(2)
df3["lag_3"] = df3.groupby("route_id")["target_1h"].shift(3)
df3["hour"] = df3["timestamp"].dt.hour
df3["dayofweek"] = df3["timestamp"].dt.dayofweek

df3 = df3.dropna().reset_index(drop=True)

split_date = df3["timestamp"].quantile(0.8)

train_df = df3[df3["timestamp"] <= split_date]
val_df   = df3[df3["timestamp"] > split_date]


features = [
    "status_1", "status_2", "status_3",
    "status_4", "status_5", "status_6",
    "lag_1", "lag_2", "lag_3",
    "hour", "dayofweek"
]

target = "target_1h"

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

In [ ]:
import lightgbm as lgb
import numpy as np

model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

pred_val = model.predict(X_val)

def metric(y_true, y_pred):
    wape = np.sum(np.abs(y_pred - y_true)) / np.sum(y_true)
    rbias = np.abs(y_pred.sum() / y_true.sum() - 1)
    return wape + rbias

score = metric(y_val.values, pred_val)
print("Validation score:", score)

1 собмит

Честный конвейер отправки данных


In [14]:
import json
import numpy as np
import pandas as pd

from forecast_pipeline import (
    apply_target_aggregates,
    build_target_aggregates,
    competition_metric,
    fit_and_predict,
    make_time_features,
    make_validation_split,
    train_full_and_predict_test,
)


In [15]:
train_df = pd.read_parquet("train_solo_track.parquet")
test_df = pd.read_parquet("test_solo_track.parquet")

train_df = make_time_features(train_df)
test_df = make_time_features(test_df)

fit_df, val_df = make_validation_split(train_df, val_horizon=8)
agg_tables, global_mean = build_target_aggregates(fit_df)

fit_df_agg = apply_target_aggregates(fit_df, agg_tables, global_mean)
val_df_agg = apply_target_aggregates(val_df, agg_tables, global_mean)

print(f"fit shape: {fit_df.shape}")
print(f"val shape: {val_df.shape}")
print("validation timestamps:")
print(sorted(val_df["timestamp"].unique()))


fit shape: (4622000, 18)
val shape: (8000, 18)
validation timestamps:
[Timestamp('2025-11-01 07:00:00'), Timestamp('2025-11-01 07:30:00'), Timestamp('2025-11-01 08:00:00'), Timestamp('2025-11-01 08:30:00'), Timestamp('2025-11-01 09:00:00'), Timestamp('2025-11-01 09:30:00'), Timestamp('2025-11-01 10:00:00'), Timestamp('2025-11-01 10:30:00')]


In [16]:
base_features = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

agg_features = [
    "target_mean_route",
    "target_mean_route_hour",
    "target_mean_route_dow",
    "target_mean_route_hour_dow",
    "target_mean_global_hour",
    "target_mean_global_dow",
]

target = "target_1h"


In [23]:
baseline_model, baseline_pred = fit_and_predict(
    fit_df,
    val_df,
    feature_cols=base_features,
)

agg_model, agg_pred = fit_and_predict(
    fit_df_agg,
    val_df_agg,
    feature_cols=base_features + agg_features,
)

y_val = val_df[target].to_numpy()

metrics = {
    "baseline": competition_metric(y_val, baseline_pred),
    "with_aggregates": competition_metric(y_val, agg_pred),
}

print(json.dumps(metrics, indent=2))


[100]	valid_0's l1: 97648
[200]	valid_0's l1: 97024
[300]	valid_0's l1: 96524.5
[400]	valid_0's l1: 96315.9
[500]	valid_0's l1: 96024.6
[600]	valid_0's l1: 95805.1
[700]	valid_0's l1: 95728.8
[800]	valid_0's l1: 95583.2
[900]	valid_0's l1: 95495.5
[1000]	valid_0's l1: 95451.8
[1100]	valid_0's l1: 95365.8
[1200]	valid_0's l1: 95292
[1300]	valid_0's l1: 95233.8
[1400]	valid_0's l1: 95169.6
[1500]	valid_0's l1: 95009.3
[1600]	valid_0's l1: 94947.6
[1700]	valid_0's l1: 94891.2
[1800]	valid_0's l1: 94869
[1900]	valid_0's l1: 94823.5
[2000]	valid_0's l1: 94796.9
[100]	valid_0's l1: 97816.2
{
  "baseline": {
    "wape": 0.3632793794442409,
    "relative_bias": 0.030253887157335794,
    "score": 0.39353326660157667
  },
  "with_aggregates": {
    "wape": 0.37418542382754244,
    "relative_bias": 0.020976065639564355,
    "score": 0.3951614894671068
  }
}


Что означают эти агрегаты:

- `target_mean_route`: средний уровень отгрузок по маршруту
- `target_mean_route_hour`: профиль маршрута по получасовому слоту
- `target_mean_route_dow`: профиль маршрута по дню недели
- `target_mean_route_hour_dow`: самый локальный недельный паттерн маршрута
- глобальные средние нужны как fallback для редких или отсутствующих комбинаций


In [ ]:
# Финальный шаг: обучаем модель на всём train и готовим предсказания для test.
full_agg_tables, full_global_mean = build_target_aggregates(train_df)

# Добавляем те же агрегатные признаки и в train, и в test.
train_full = apply_target_aggregates(train_df, full_agg_tables, full_global_mean)
test_full = apply_target_aggregates(test_df, full_agg_tables, full_global_mean)

# Обучаем финальную модель на всём train и получаем предсказания для test.
final_model, pred_test = train_full_and_predict_test(
    train_full,
    test_full,
    feature_cols=base_features + agg_features,
)

# Объём отгрузок не может быть отрицательным, поэтому обрезаем отрицательные значения до нуля.
pred_test = np.clip(pred_test, a_min=0, a_max=None)


In [ ]:
# Формируем итоговый файл сабмита для платформы.
# Важно: платформа ждёт ровно две колонки: id и y_pred.
submission = pd.DataFrame({
    "id": test_df["id"],
    "y_pred": pred_test,
})

# Даём файлу версионное имя, чтобы не путать его с предыдущими сабмитами.
submission_path = "submission_route_time_agg_v2.csv"
submission.to_csv(submission_path, index=False)

print(f"Сабмит сохранён в файл: {submission_path}")
submission.head()


In [ ]:
# Быстрая проверка распределения предсказаний. Это не score, а только sanity check.
submission["y_pred"].describe()


Что делать после формирования сабмита:

1. Загрузить `submission_route_time_agg_v2.csv` на платформу
2. После расчёта leaderboard-score вручную записать его в заметку ниже
3. Сравнить новый score с предыдущим лучшим результатом `0.365`


Следующий эксперимент: добавляем count-агрегаты.

Идея: среднее по группе может быть шумным, если наблюдений мало.
Поэтому мы даём модели ещё и количество наблюдений для каждой агрегатной группы.


In [17]:
# Считаем count-агрегаты только по обучающей части, чтобы не было утечки.
route_count = (
    fit_df.groupby(["route_id"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_count"})
)

route_hour_count = (
    fit_df.groupby(["route_id", "hour_slot"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_hour_count"})
)

route_dow_count = (
    fit_df.groupby(["route_id", "dayofweek"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_dow_count"})
)

route_hour_dow_count = (
    fit_df.groupby(["route_id", "hour_slot", "dayofweek"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_hour_dow_count"})
)


In [18]:
# Добавляем count-агрегаты в train и validation.
fit_df_count = fit_df_agg.merge(route_count, on=["route_id"], how="left")
fit_df_count = fit_df_count.merge(route_hour_count, on=["route_id", "hour_slot"], how="left")
fit_df_count = fit_df_count.merge(route_dow_count, on=["route_id", "dayofweek"], how="left")
fit_df_count = fit_df_count.merge(
    route_hour_dow_count,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

val_df_count = val_df_agg.merge(route_count, on=["route_id"], how="left")
val_df_count = val_df_count.merge(route_hour_count, on=["route_id", "hour_slot"], how="left")
val_df_count = val_df_count.merge(route_dow_count, on=["route_id", "dayofweek"], how="left")
val_df_count = val_df_count.merge(
    route_hour_dow_count,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

count_features = [
    "route_count",
    "route_hour_count",
    "route_dow_count",
    "route_hour_dow_count",
]

for col in count_features:
    fit_df_count[col] = fit_df_count[col].fillna(0)
    val_df_count[col] = val_df_count[col].fillna(0)


In [22]:
# Сначала заново считаем локальную метрику для текущей модели без count-признаков.
_, agg_pred_local = fit_and_predict(
    fit_df_agg,
    val_df_agg,
    feature_cols=base_features + agg_features,
)

# Потом считаем локальную метрику для модели с count-признаками.
_, count_pred = fit_and_predict(
    fit_df_count,
    val_df_count,
    feature_cols=base_features + agg_features + count_features,
)

y_val = val_df[target].to_numpy()
agg_metrics_local = competition_metric(y_val, agg_pred_local)
count_metrics = competition_metric(y_val, count_pred)

print("Текущая модель с агрегатами:")
print(json.dumps(agg_metrics_local, indent=2, ensure_ascii=False))
print()
print("Модель с агрегатами и count-признаками:")
print(json.dumps(count_metrics, indent=2, ensure_ascii=False))
print()
print("Скор текущей модели:", agg_metrics_local["score"])
print("Скор модели с count:", count_metrics["score"])


[100]	valid_0's l1: 97816.2
[100]	valid_0's l1: 97572.4
[200]	valid_0's l1: 97138
[300]	valid_0's l1: 96961.8
[400]	valid_0's l1: 96911.1
[500]	valid_0's l1: 96869
[600]	valid_0's l1: 96844.1
[700]	valid_0's l1: 96817.1
[800]	valid_0's l1: 96782.8
Текущая модель с агрегатами:
{
  "wape": 0.37418542382754244,
  "relative_bias": 0.020976065639564355,
  "score": 0.3951614894671068
}

Модель с агрегатами и count-признаками:
{
  "wape": 0.3708017821356842,
  "relative_bias": 0.019736309836413786,
  "score": 0.390538091972098
}

Скор текущей модели: 0.3951614894671068
Скор модели с count: 0.390538091972098


Как интерпретировать результат этого эксперимента:

- если `score` стал меньше, count-признаки полезны
- если `score` почти не изменился, можно попробовать `median`
- если `score` стал хуже, значит count в таком виде модели не помогает


Финальный сабмит для версии с count-агрегатами.

Этот блок нужно запускать только если локально вариант с count выглядит не хуже или лучше.


In [23]:
# Считаем те же count-агрегаты уже по всему train для финального сабмита.
full_route_count = (
    train_df.groupby(["route_id"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_count"})
)

full_route_hour_count = (
    train_df.groupby(["route_id", "hour_slot"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_hour_count"})
)

full_route_dow_count = (
    train_df.groupby(["route_id", "dayofweek"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_dow_count"})
)

full_route_hour_dow_count = (
    train_df.groupby(["route_id", "hour_slot", "dayofweek"], as_index=False)[target]
    .size()
    .rename(columns={"size": "route_hour_dow_count"})
)


In [24]:
# Добавляем mean- и count-признаки в полный train и test.
train_full_count = train_full.merge(full_route_count, on=["route_id"], how="left")
train_full_count = train_full_count.merge(full_route_hour_count, on=["route_id", "hour_slot"], how="left")
train_full_count = train_full_count.merge(full_route_dow_count, on=["route_id", "dayofweek"], how="left")
train_full_count = train_full_count.merge(
    full_route_hour_dow_count,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

test_full_count = test_full.merge(full_route_count, on=["route_id"], how="left")
test_full_count = test_full_count.merge(full_route_hour_count, on=["route_id", "hour_slot"], how="left")
test_full_count = test_full_count.merge(full_route_dow_count, on=["route_id", "dayofweek"], how="left")
test_full_count = test_full_count.merge(
    full_route_hour_dow_count,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

for col in count_features:
    train_full_count[col] = train_full_count[col].fillna(0)
    test_full_count[col] = test_full_count[col].fillna(0)


In [25]:
# Обучаем финальную модель для count-версии и сохраняем отдельный файл сабмита.
final_count_model, pred_test_count = train_full_and_predict_test(
    train_full_count,
    test_full_count,
    feature_cols=base_features + agg_features + count_features,
)

pred_test_count = np.clip(pred_test_count, a_min=0, a_max=None)

submission_count = pd.DataFrame({
    "id": test_df["id"],
    "y_pred": pred_test_count,
})

submission_count_path = "submission_route_time_agg_count_v1.csv"
submission_count.to_csv(submission_count_path, index=False)

print(f"Сабмит для count-версии сохранён в файл: {submission_count_path}")
submission_count.head()


Сабмит для count-версии сохранён в файл: submission_route_time_agg_count_v1.csv


,id,y_pred
0,0,171144.705778
1,1,168437.042467
2,2,140738.058902
3,3,138361.580539
4,4,160615.902489


Заметка для презентации по count-версии:

- Что добавили: count-агрегаты по тем же группам, где уже были mean-агрегаты
- Зачем: показать модели, насколько надёжно посчитано среднее по группе
- Имя файла сабмита: `submission_route_time_agg_count_v1.csv`
- Leaderboard score: 0.36776659947433227
- Вывод: стало на 3 тысячных хуже, но это паблик лидерборд так что может быть влияние всё таки будет 


Самостоятельный эксперимент: median-агрегаты.

Этот блок можно запускать отдельно после перезапуска kernel.
Он сам делает импорты, загружает данные, считает признаки, валидируется и готовит отдельный сабмит.


In [ ]:
import json
import numpy as np
import pandas as pd

from forecast_pipeline import (
    apply_target_aggregates,
    build_target_aggregates,
    competition_metric,
    fit_and_predict,
    make_time_features,
    make_validation_split,
    train_full_and_predict_test,
)


In [ ]:
# Загружаем train и test заново, чтобы блок был самодостаточным.
train_df_median = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_median = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_median = make_time_features(train_df_median)
test_df_median = make_time_features(test_df_median)

# Делаем честный split: последние 8 таймстампов идут в validation.
fit_df_median, val_df_median = make_validation_split(train_df_median, val_horizon=8)

# Сначала считаем уже рабочие mean-агрегаты.
agg_tables_median, global_mean_median = build_target_aggregates(fit_df_median)
fit_df_mean = apply_target_aggregates(fit_df_median, agg_tables_median, global_mean_median)
val_df_mean = apply_target_aggregates(val_df_median, agg_tables_median, global_mean_median)


In [ ]:
# Базовые признаки из route_id и timestamp.
base_features_median = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

# Mean-агрегаты из лучшей текущей версии.
agg_features_median = [
    "target_mean_route",
    "target_mean_route_hour",
    "target_mean_route_dow",
    "target_mean_route_hour_dow",
    "target_mean_global_hour",
    "target_mean_global_dow",
]

target_median_col = "target_1h"


In [ ]:
# Считаем median-агрегаты только по обучающей части, чтобы не было утечки.
median_route = (
    fit_df_median.groupby(["route_id"], as_index=False)[target_median_col]
    .median()
    .rename(columns={target_median_col: "target_median_route"})
)

median_route_hour = (
    fit_df_median.groupby(["route_id", "hour_slot"], as_index=False)[target_median_col]
    .median()
    .rename(columns={target_median_col: "target_median_route_hour"})
)

median_route_dow = (
    fit_df_median.groupby(["route_id", "dayofweek"], as_index=False)[target_median_col]
    .median()
    .rename(columns={target_median_col: "target_median_route_dow"})
)

median_route_hour_dow = (
    fit_df_median.groupby(["route_id", "hour_slot", "dayofweek"], as_index=False)[target_median_col]
    .median()
    .rename(columns={target_median_col: "target_median_route_hour_dow"})
)


In [ ]:
# Добавляем median-агрегаты в train и validation.
fit_df_median_features = fit_df_mean.merge(median_route, on=["route_id"], how="left")
fit_df_median_features = fit_df_median_features.merge(median_route_hour, on=["route_id", "hour_slot"], how="left")
fit_df_median_features = fit_df_median_features.merge(median_route_dow, on=["route_id", "dayofweek"], how="left")
fit_df_median_features = fit_df_median_features.merge(
    median_route_hour_dow,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

val_df_median_features = val_df_mean.merge(median_route, on=["route_id"], how="left")
val_df_median_features = val_df_median_features.merge(median_route_hour, on=["route_id", "hour_slot"], how="left")
val_df_median_features = val_df_median_features.merge(median_route_dow, on=["route_id", "dayofweek"], how="left")
val_df_median_features = val_df_median_features.merge(
    median_route_hour_dow,
    on=["route_id", "hour_slot", "dayofweek"],
    how="left",
)

median_features = [
    "target_median_route",
    "target_median_route_hour",
    "target_median_route_dow",
    "target_median_route_hour_dow",
]

# Если точной median-комбинации нет, откатываемся к более общим median-признакам.
fit_df_median_features["target_median_route"] = fit_df_median_features["target_median_route"].fillna(global_mean_median)
val_df_median_features["target_median_route"] = val_df_median_features["target_median_route"].fillna(global_mean_median)

fit_df_median_features["target_median_route_hour"] = fit_df_median_features["target_median_route_hour"].fillna(fit_df_median_features["target_median_route"])
val_df_median_features["target_median_route_hour"] = val_df_median_features["target_median_route_hour"].fillna(val_df_median_features["target_median_route"])

fit_df_median_features["target_median_route_dow"] = fit_df_median_features["target_median_route_dow"].fillna(fit_df_median_features["target_median_route"])
val_df_median_features["target_median_route_dow"] = val_df_median_features["target_median_route_dow"].fillna(val_df_median_features["target_median_route"])

fit_df_median_features["target_median_route_hour_dow"] = fit_df_median_features["target_median_route_hour_dow"].fillna(fit_df_median_features["target_median_route_hour"])
val_df_median_features["target_median_route_hour_dow"] = val_df_median_features["target_median_route_hour_dow"].fillna(val_df_median_features["target_median_route_hour"])


In [ ]:
# Сравниваем текущую лучшую mean-версию и новую mean+median версию.
_, pred_mean_local = fit_and_predict(
    fit_df_mean,
    val_df_mean,
    feature_cols=base_features_median + agg_features_median,
)

_, pred_median_local = fit_and_predict(
    fit_df_median_features,
    val_df_median_features,
    feature_cols=base_features_median + agg_features_median + median_features,
)

y_val_median = val_df_median[target_median_col].to_numpy()
mean_metrics_local = competition_metric(y_val_median, pred_mean_local)
median_metrics_local = competition_metric(y_val_median, pred_median_local)

print("Текущая модель с mean-агрегатами:")
print(json.dumps(mean_metrics_local, indent=2, ensure_ascii=False))
print()
print("Модель с mean- и median-агрегатами:")
print(json.dumps(median_metrics_local, indent=2, ensure_ascii=False))
print()
print("Скор текущей модели:", mean_metrics_local["score"])
print("Скор модели с median:", median_metrics_local["score"])


Вывод по median-эксперименту:

- Локальный score текущей модели: `0.3951614894671068`
- Локальный score модели с median: `0.40979012962272854`
- Результат: стало хуже
- Вывод: median-агрегаты в текущем виде не подходят, финальный сабмит для этой версии не делаем


Следующий эксперимент: сглаженные mean-агрегаты.

Идея: обычное среднее по группе может быть шумным, если наблюдений мало.
Поэтому вместо сырых средних попробуем сглаженные средние, которые тянутся к глобальному среднему для редких групп.


In [32]:
import json
import numpy as np
import pandas as pd

from forecast_pipeline import (
    competition_metric,
    fit_and_predict,
    make_time_features,
    make_validation_split,
    train_full_and_predict_test,
)


In [33]:
# Загружаем train и test заново, чтобы блок был полностью самодостаточным.
train_df_smooth = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_smooth = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_smooth = make_time_features(train_df_smooth)
test_df_smooth = make_time_features(test_df_smooth)

# Последние 8 таймстампов используем как validation.
fit_df_smooth, val_df_smooth = make_validation_split(train_df_smooth, val_horizon=8)

base_features_smooth = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_smooth_col = "target_1h"
alpha = 20.0


In [34]:
# Считаем сглаженные агрегаты по обучающей части.
global_mean_smooth = float(fit_df_smooth[target_smooth_col].mean())

route_stats = fit_df_smooth.groupby(["route_id"])[target_smooth_col].agg(["mean", "size"]).reset_index()
route_stats = route_stats.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats["target_smooth_route"] = (route_stats["route_mean"] * route_stats["route_count"] + global_mean_smooth * alpha) / (route_stats["route_count"] + alpha)

route_hour_stats = fit_df_smooth.groupby(["route_id", "hour_slot"])[target_smooth_col].agg(["mean", "size"]).reset_index()
route_hour_stats = route_hour_stats.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats["target_smooth_route_hour"] = (route_hour_stats["route_hour_mean"] * route_hour_stats["route_hour_count"] + global_mean_smooth * alpha) / (route_hour_stats["route_hour_count"] + alpha)

route_dow_stats = fit_df_smooth.groupby(["route_id", "dayofweek"])[target_smooth_col].agg(["mean", "size"]).reset_index()
route_dow_stats = route_dow_stats.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats["target_smooth_route_dow"] = (route_dow_stats["route_dow_mean"] * route_dow_stats["route_dow_count"] + global_mean_smooth * alpha) / (route_dow_stats["route_dow_count"] + alpha)

route_hour_dow_stats = fit_df_smooth.groupby(["route_id", "hour_slot", "dayofweek"])[target_smooth_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats = route_hour_dow_stats.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats["target_smooth_route_hour_dow"] = (route_hour_dow_stats["route_hour_dow_mean"] * route_hour_dow_stats["route_hour_dow_count"] + global_mean_smooth * alpha) / (route_hour_dow_stats["route_hour_dow_count"] + alpha)


In [35]:
# Добавляем сглаженные признаки в train и validation.
fit_df_smooth_features = fit_df_smooth.merge(route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
fit_df_smooth_features = fit_df_smooth_features.merge(route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
fit_df_smooth_features = fit_df_smooth_features.merge(route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
fit_df_smooth_features = fit_df_smooth_features.merge(route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

val_df_smooth_features = val_df_smooth.merge(route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
val_df_smooth_features = val_df_smooth_features.merge(route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
val_df_smooth_features = val_df_smooth_features.merge(route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
val_df_smooth_features = val_df_smooth_features.merge(route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

smooth_features = ["target_smooth_route", "target_smooth_route_hour", "target_smooth_route_dow", "target_smooth_route_hour_dow"]

fit_df_smooth_features["target_smooth_route"] = fit_df_smooth_features["target_smooth_route"].fillna(global_mean_smooth)
val_df_smooth_features["target_smooth_route"] = val_df_smooth_features["target_smooth_route"].fillna(global_mean_smooth)
fit_df_smooth_features["target_smooth_route_hour"] = fit_df_smooth_features["target_smooth_route_hour"].fillna(fit_df_smooth_features["target_smooth_route"])
val_df_smooth_features["target_smooth_route_hour"] = val_df_smooth_features["target_smooth_route_hour"].fillna(val_df_smooth_features["target_smooth_route"])
fit_df_smooth_features["target_smooth_route_dow"] = fit_df_smooth_features["target_smooth_route_dow"].fillna(fit_df_smooth_features["target_smooth_route"])
val_df_smooth_features["target_smooth_route_dow"] = val_df_smooth_features["target_smooth_route_dow"].fillna(val_df_smooth_features["target_smooth_route"])
fit_df_smooth_features["target_smooth_route_hour_dow"] = fit_df_smooth_features["target_smooth_route_hour_dow"].fillna(fit_df_smooth_features["target_smooth_route_hour"])
val_df_smooth_features["target_smooth_route_hour_dow"] = val_df_smooth_features["target_smooth_route_hour_dow"].fillna(val_df_smooth_features["target_smooth_route_hour"])


In [36]:
# Сравниваем baseline и smoothing-версию локально.
_, pred_smooth_local = fit_and_predict(
    fit_df_smooth_features,
    val_df_smooth_features,
    feature_cols=base_features_smooth + smooth_features,
)

y_val_smooth = val_df_smooth[target_smooth_col].to_numpy()
smooth_metrics = competition_metric(y_val_smooth, pred_smooth_local)

print("Текущий ориентир по leaderboard: 0.365")
print("Локальный score baseline для сравнения: 0.3951614894671068")
print("Локальный score smoothing-версии:", smooth_metrics["score"])
print(json.dumps(smooth_metrics, indent=2, ensure_ascii=False))


[100]	valid_0's l1: 97514.2
[200]	valid_0's l1: 97040
[300]	valid_0's l1: 96975.6
Текущий ориентир по leaderboard: 0.365
Локальный score baseline для сравнения: 0.3951614894671068
Локальный score smoothing-версии: 0.3835836125379652
{
  "wape": 0.37150618700052895,
  "relative_bias": 0.01207742553743623,
  "score": 0.3835836125379652
}


Как интерпретировать результат smoothing-эксперимента:

- если локальный `score` стал меньше 0.3951614894671068, smoothing выглядит перспективно
- если локальный `score` почти не изменился, можно пробовать другой `alpha`
- если стало хуже, следующий шаг — CatBoost или blend


Финальный сабмит для версии со сглаженными агрегатами.

Этот блок запускайте только если локально smoothing выглядит не хуже baseline.


In [37]:
# Считаем сглаженные агрегаты уже по всему train для финального сабмита.
global_mean_full = float(train_df_smooth[target_smooth_col].mean())

full_route_stats = train_df_smooth.groupby(["route_id"])[target_smooth_col].agg(["mean", "size"]).reset_index()
full_route_stats = full_route_stats.rename(columns={"mean": "route_mean", "size": "route_count"})
full_route_stats["target_smooth_route"] = (full_route_stats["route_mean"] * full_route_stats["route_count"] + global_mean_full * alpha) / (full_route_stats["route_count"] + alpha)

full_route_hour_stats = train_df_smooth.groupby(["route_id", "hour_slot"])[target_smooth_col].agg(["mean", "size"]).reset_index()
full_route_hour_stats = full_route_hour_stats.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
full_route_hour_stats["target_smooth_route_hour"] = (full_route_hour_stats["route_hour_mean"] * full_route_hour_stats["route_hour_count"] + global_mean_full * alpha) / (full_route_hour_stats["route_hour_count"] + alpha)

full_route_dow_stats = train_df_smooth.groupby(["route_id", "dayofweek"])[target_smooth_col].agg(["mean", "size"]).reset_index()
full_route_dow_stats = full_route_dow_stats.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
full_route_dow_stats["target_smooth_route_dow"] = (full_route_dow_stats["route_dow_mean"] * full_route_dow_stats["route_dow_count"] + global_mean_full * alpha) / (full_route_dow_stats["route_dow_count"] + alpha)

full_route_hour_dow_stats = train_df_smooth.groupby(["route_id", "hour_slot", "dayofweek"])[target_smooth_col].agg(["mean", "size"]).reset_index()
full_route_hour_dow_stats = full_route_hour_dow_stats.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
full_route_hour_dow_stats["target_smooth_route_hour_dow"] = (full_route_hour_dow_stats["route_hour_dow_mean"] * full_route_hour_dow_stats["route_hour_dow_count"] + global_mean_full * alpha) / (full_route_hour_dow_stats["route_hour_dow_count"] + alpha)


In [38]:
# Добавляем сглаженные признаки в полный train и test.
train_full_smooth = train_df_smooth.merge(full_route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
train_full_smooth = train_full_smooth.merge(full_route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
train_full_smooth = train_full_smooth.merge(full_route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
train_full_smooth = train_full_smooth.merge(full_route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

test_full_smooth = test_df_smooth.merge(full_route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
test_full_smooth = test_full_smooth.merge(full_route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
test_full_smooth = test_full_smooth.merge(full_route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
test_full_smooth = test_full_smooth.merge(full_route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

train_full_smooth["target_smooth_route"] = train_full_smooth["target_smooth_route"].fillna(global_mean_full)
test_full_smooth["target_smooth_route"] = test_full_smooth["target_smooth_route"].fillna(global_mean_full)
train_full_smooth["target_smooth_route_hour"] = train_full_smooth["target_smooth_route_hour"].fillna(train_full_smooth["target_smooth_route"])
test_full_smooth["target_smooth_route_hour"] = test_full_smooth["target_smooth_route_hour"].fillna(test_full_smooth["target_smooth_route"])
train_full_smooth["target_smooth_route_dow"] = train_full_smooth["target_smooth_route_dow"].fillna(train_full_smooth["target_smooth_route"])
test_full_smooth["target_smooth_route_dow"] = test_full_smooth["target_smooth_route_dow"].fillna(test_full_smooth["target_smooth_route"])
train_full_smooth["target_smooth_route_hour_dow"] = train_full_smooth["target_smooth_route_hour_dow"].fillna(train_full_smooth["target_smooth_route_hour"])
test_full_smooth["target_smooth_route_hour_dow"] = test_full_smooth["target_smooth_route_hour_dow"].fillna(test_full_smooth["target_smooth_route_hour"])


In [39]:
# Обучаем финальную smoothing-модель и сохраняем отдельный файл сабмита.
final_smooth_model, pred_test_smooth = train_full_and_predict_test(
    train_full_smooth,
    test_full_smooth,
    feature_cols=base_features_smooth + smooth_features,
)

pred_test_smooth = np.clip(pred_test_smooth, a_min=0, a_max=None)

submission_smooth = pd.DataFrame({
    "id": test_df_smooth["id"],
    "y_pred": pred_test_smooth,
})

submission_smooth_path = "submission_route_time_agg_smooth_v1.csv"
submission_smooth.to_csv(submission_smooth_path, index=False)

print(f"Сабмит для smoothing-версии сохранён в файл: {submission_smooth_path}")
submission_smooth.head()


Сабмит для smoothing-версии сохранён в файл: submission_route_time_agg_smooth_v1.csv


,id,y_pred
0,0,184380.332036
1,1,176575.090364
2,2,153351.967834
3,3,146711.327994
4,4,171841.538427


Заметка для презентации по smoothing-версии:

- Что добавили: сглаженные mean-агрегаты вместо сырых средних
- Зачем: сделать агрегаты устойчивее для редких групп
- Параметр сглаживания: `alpha = 20`
- Имя файла сабмита: `submission_route_time_agg_smooth_v1.csv`
- Локальный score: 0.3835836125379652
- Leaderboard score: 0.3608341250041473
- Вывод: до этого было 0.3654444629699353 стало лучше 


Уточняющий подбор `alpha` вокруг лучшего значения.

В прошлом прогоне лучшим оказался `alpha = 10`, поэтому теперь проверяем ближайшие значения.


In [ ]:
# Загружаем данные заново, чтобы блок был полностью самостоятельным.
train_df_alpha = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")

train_df_alpha = make_time_features(train_df_alpha)
fit_df_alpha, val_df_alpha = make_validation_split(train_df_alpha, val_horizon=8)

base_features_alpha = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_alpha_col = "target_1h"
alpha_values = [7, 8, 9, 10, 11, 12, 13]
y_val_alpha = val_df_alpha[target_alpha_col].to_numpy()


In [ ]:
# Загружаем данные заново, чтобы блок был полностью самостоятельным.
train_df_alpha = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")

train_df_alpha = make_time_features(train_df_alpha)
fit_df_alpha, val_df_alpha = make_validation_split(train_df_alpha, val_horizon=8)

base_features_alpha = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_alpha_col = "target_1h"
alpha_values = [5, 10, 20, 30, 50]
y_val_alpha = val_df_alpha[target_alpha_col].to_numpy()


In [ ]:
# Функция для построения сглаженных признаков для заданного alpha.
def build_smoothed_frames(fit_frame, val_frame, alpha_value):
    global_mean = float(fit_frame[target_alpha_col].mean())

    route_stats = fit_frame.groupby(["route_id"])[target_alpha_col].agg(["mean", "size"]).reset_index()
    route_stats = route_stats.rename(columns={"mean": "route_mean", "size": "route_count"})
    route_stats["target_smooth_route"] = (route_stats["route_mean"] * route_stats["route_count"] + global_mean * alpha_value) / (route_stats["route_count"] + alpha_value)

    route_hour_stats = fit_frame.groupby(["route_id", "hour_slot"])[target_alpha_col].agg(["mean", "size"]).reset_index()
    route_hour_stats = route_hour_stats.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
    route_hour_stats["target_smooth_route_hour"] = (route_hour_stats["route_hour_mean"] * route_hour_stats["route_hour_count"] + global_mean * alpha_value) / (route_hour_stats["route_hour_count"] + alpha_value)

    route_dow_stats = fit_frame.groupby(["route_id", "dayofweek"])[target_alpha_col].agg(["mean", "size"]).reset_index()
    route_dow_stats = route_dow_stats.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
    route_dow_stats["target_smooth_route_dow"] = (route_dow_stats["route_dow_mean"] * route_dow_stats["route_dow_count"] + global_mean * alpha_value) / (route_dow_stats["route_dow_count"] + alpha_value)

    route_hour_dow_stats = fit_frame.groupby(["route_id", "hour_slot", "dayofweek"])[target_alpha_col].agg(["mean", "size"]).reset_index()
    route_hour_dow_stats = route_hour_dow_stats.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
    route_hour_dow_stats["target_smooth_route_hour_dow"] = (route_hour_dow_stats["route_hour_dow_mean"] * route_hour_dow_stats["route_hour_dow_count"] + global_mean * alpha_value) / (route_hour_dow_stats["route_hour_dow_count"] + alpha_value)

    fit_features = fit_frame.merge(route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
    fit_features = fit_features.merge(route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
    fit_features = fit_features.merge(route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
    fit_features = fit_features.merge(route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

    val_features = val_frame.merge(route_stats[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
    val_features = val_features.merge(route_hour_stats[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
    val_features = val_features.merge(route_dow_stats[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
    val_features = val_features.merge(route_hour_dow_stats[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

    for frame in (fit_features, val_features):
        frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean)
        frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
        frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
        frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])

    return fit_features, val_features


In [ ]:
# Прогоняем несколько alpha и сравниваем локальный score.
results = []
smooth_features_alpha = [
    "target_smooth_route",
    "target_smooth_route_hour",
    "target_smooth_route_dow",
    "target_smooth_route_hour_dow",
]

for alpha_value in alpha_values:
    fit_features_alpha, val_features_alpha = build_smoothed_frames(
        fit_df_alpha,
        val_df_alpha,
        alpha_value,
    )

    _, pred_alpha = fit_and_predict(
        fit_features_alpha,
        val_features_alpha,
        feature_cols=base_features_alpha + smooth_features_alpha,
    )

    metrics_alpha = competition_metric(y_val_alpha, pred_alpha)
    results.append({
        "alpha": alpha_value,
        "wape": metrics_alpha["wape"],
        "relative_bias": metrics_alpha["relative_bias"],
        "score": metrics_alpha["score"],
    })

results_df = pd.DataFrame(results).sort_values("score")
results_df


In [ ]:
# Показываем лучшую alpha и сравниваем её с текущим ориентиром.
best_row = results_df.iloc[0]
print("Текущий локальный ориентир smoothing-версии:", 0.3835836125379652)
print("Лучшая alpha:", best_row["alpha"])
print("Лучший локальный score:", best_row["score"])
print("Полная таблица результатов:")
print(results_df.to_string(index=False))


Вывод по подбору `alpha`:

- Лучшее значение на локальной валидации: `alpha = 9`
- Лучший локальный score: `0.37880477073049845`
- Это лучше, чем текущий локальный ориентир `0.3835836125379652`
- Следующий шаг: делаем новый сабмит именно с `alpha = 9`


Заметка для презентации по подбору `alpha`:

- Что делали: подбирали силу сглаживания для исторических агрегатов
- Проверенные значения: `7, 8, 9, 10, 11, 12, 13`
- Лучший `alpha`: `9`
- Лучший локальный score: `0.37880477073049845`
- Решение: формируем отдельный сабмит с `alpha = 9`


Финальный сабмит для smoothing-версии с `alpha = 9`.

Этот блок можно запускать отдельно после любого перезапуска kernel.
Он заново загружает данные, строит сглаженные признаки с `alpha = 9` и сохраняет отдельный файл сабмита.


In [55]:
import numpy as np
import pandas as pd

from forecast_pipeline import (
    make_time_features,
    train_full_and_predict_test,
)


In [56]:
# Загружаем train и test заново для финального сабмита с alpha = 9.
train_df_alpha9 = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_alpha9 = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_alpha9 = make_time_features(train_df_alpha9)
test_df_alpha9 = make_time_features(test_df_alpha9)

base_features_alpha9 = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

alpha_best = 9.0


In [57]:
# Считаем сглаженные агрегаты по всему train с лучшим alpha = 9.
global_mean_alpha9 = float(train_df_alpha9["target_1h"].mean())

route_stats_alpha9 = train_df_alpha9.groupby(["route_id"])["target_1h"].agg(["mean", "size"]).reset_index()
route_stats_alpha9 = route_stats_alpha9.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_alpha9["target_smooth_route"] = (route_stats_alpha9["route_mean"] * route_stats_alpha9["route_count"] + global_mean_alpha9 * alpha_best) / (route_stats_alpha9["route_count"] + alpha_best)

route_hour_stats_alpha9 = train_df_alpha9.groupby(["route_id", "hour_slot"])["target_1h"].agg(["mean", "size"]).reset_index()
route_hour_stats_alpha9 = route_hour_stats_alpha9.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_alpha9["target_smooth_route_hour"] = (route_hour_stats_alpha9["route_hour_mean"] * route_hour_stats_alpha9["route_hour_count"] + global_mean_alpha9 * alpha_best) / (route_hour_stats_alpha9["route_hour_count"] + alpha_best)

route_dow_stats_alpha9 = train_df_alpha9.groupby(["route_id", "dayofweek"])["target_1h"].agg(["mean", "size"]).reset_index()
route_dow_stats_alpha9 = route_dow_stats_alpha9.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_alpha9["target_smooth_route_dow"] = (route_dow_stats_alpha9["route_dow_mean"] * route_dow_stats_alpha9["route_dow_count"] + global_mean_alpha9 * alpha_best) / (route_dow_stats_alpha9["route_dow_count"] + alpha_best)

route_hour_dow_stats_alpha9 = train_df_alpha9.groupby(["route_id", "hour_slot", "dayofweek"])["target_1h"].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_alpha9 = route_hour_dow_stats_alpha9.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_alpha9["target_smooth_route_hour_dow"] = (route_hour_dow_stats_alpha9["route_hour_dow_mean"] * route_hour_dow_stats_alpha9["route_hour_dow_count"] + global_mean_alpha9 * alpha_best) / (route_hour_dow_stats_alpha9["route_hour_dow_count"] + alpha_best)


In [58]:
# Добавляем сглаженные признаки в train и test.
train_full_alpha9 = train_df_alpha9.merge(route_stats_alpha9[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
train_full_alpha9 = train_full_alpha9.merge(route_hour_stats_alpha9[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
train_full_alpha9 = train_full_alpha9.merge(route_dow_stats_alpha9[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
train_full_alpha9 = train_full_alpha9.merge(route_hour_dow_stats_alpha9[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

test_full_alpha9 = test_df_alpha9.merge(route_stats_alpha9[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
test_full_alpha9 = test_full_alpha9.merge(route_hour_stats_alpha9[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
test_full_alpha9 = test_full_alpha9.merge(route_dow_stats_alpha9[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
test_full_alpha9 = test_full_alpha9.merge(route_hour_dow_stats_alpha9[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

smooth_features_alpha9 = [
    "target_smooth_route",
    "target_smooth_route_hour",
    "target_smooth_route_dow",
    "target_smooth_route_hour_dow",
]

train_full_alpha9["target_smooth_route"] = train_full_alpha9["target_smooth_route"].fillna(global_mean_alpha9)
test_full_alpha9["target_smooth_route"] = test_full_alpha9["target_smooth_route"].fillna(global_mean_alpha9)
train_full_alpha9["target_smooth_route_hour"] = train_full_alpha9["target_smooth_route_hour"].fillna(train_full_alpha9["target_smooth_route"])
test_full_alpha9["target_smooth_route_hour"] = test_full_alpha9["target_smooth_route_hour"].fillna(test_full_alpha9["target_smooth_route"])
train_full_alpha9["target_smooth_route_dow"] = train_full_alpha9["target_smooth_route_dow"].fillna(train_full_alpha9["target_smooth_route"])
test_full_alpha9["target_smooth_route_dow"] = test_full_alpha9["target_smooth_route_dow"].fillna(test_full_alpha9["target_smooth_route"])
train_full_alpha9["target_smooth_route_hour_dow"] = train_full_alpha9["target_smooth_route_hour_dow"].fillna(train_full_alpha9["target_smooth_route_hour"])
test_full_alpha9["target_smooth_route_hour_dow"] = test_full_alpha9["target_smooth_route_hour_dow"].fillna(test_full_alpha9["target_smooth_route_hour"])


In [59]:
# Обучаем финальную модель и сохраняем отдельный сабмит с alpha = 9.
final_alpha9_model, pred_test_alpha9 = train_full_and_predict_test(
    train_full_alpha9,
    test_full_alpha9,
    feature_cols=base_features_alpha9 + smooth_features_alpha9,
)

pred_test_alpha9 = np.clip(pred_test_alpha9, a_min=0, a_max=None)

submission_alpha9 = pd.DataFrame({
    "id": test_df_alpha9["id"],
    "y_pred": pred_test_alpha9,
})

submission_alpha9_path = "submission_route_time_agg_smooth_alpha9_v1.csv"
submission_alpha9.to_csv(submission_alpha9_path, index=False)

print(f"Сабмит с alpha = 9 сохранён в файл: {submission_alpha9_path}")
submission_alpha9.head()


Сабмит с alpha = 9 сохранён в файл: submission_route_time_agg_smooth_alpha9_v1.csv


,id,y_pred
0,0,191030.037841
1,1,182371.346059
2,2,163499.139571
3,3,156323.797546
4,4,182196.956820


Заметка для презентации по версии с `alpha = 9`:

- Что выбрали: лучшую локальную силу сглаживания по результатам валидации
- Лучший `alpha`: `9`
- Лучший локальный score: `0.37880477073049845`
- Имя файла сабмита: `submission_route_time_agg_smooth_alpha9_v1.csv`
- Leaderboard score: `0.36237701635610053`
- Вывод: локально стало лучше, но на public leaderboard версия с `alpha = 20` всё равно сильнее


Следующий эксперимент: `CatBoost` на лучшем наборе признаков.

Сейчас лучшая боевая версия — smoothing с `alpha = 20`.
Теперь проверим, сможет ли другая модель на тех же честных признаках дать лучший результат.


In [ ]:
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

from forecast_pipeline import (
    competition_metric,
    make_time_features,
    make_validation_split,
)


In [ ]:
# Загружаем train и test заново, чтобы блок был полностью самостоятельным.
train_df_cat = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_cat = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_cat = make_time_features(train_df_cat)
test_df_cat = make_time_features(test_df_cat)

fit_df_cat, val_df_cat = make_validation_split(train_df_cat, val_horizon=8)

base_features_cat = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_cat_col = "target_1h"
alpha_cat = 20.0


In [ ]:
# Строим те же сглаженные признаки, что дали лучший публичный результат у LightGBM.
global_mean_cat = float(fit_df_cat[target_cat_col].mean())

route_stats_cat = fit_df_cat.groupby(["route_id"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_stats_cat = route_stats_cat.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_cat["target_smooth_route"] = (route_stats_cat["route_mean"] * route_stats_cat["route_count"] + global_mean_cat * alpha_cat) / (route_stats_cat["route_count"] + alpha_cat)

route_hour_stats_cat = fit_df_cat.groupby(["route_id", "hour_slot"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_hour_stats_cat = route_hour_stats_cat.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_cat["target_smooth_route_hour"] = (route_hour_stats_cat["route_hour_mean"] * route_hour_stats_cat["route_hour_count"] + global_mean_cat * alpha_cat) / (route_hour_stats_cat["route_hour_count"] + alpha_cat)

route_dow_stats_cat = fit_df_cat.groupby(["route_id", "dayofweek"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_dow_stats_cat = route_dow_stats_cat.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_cat["target_smooth_route_dow"] = (route_dow_stats_cat["route_dow_mean"] * route_dow_stats_cat["route_dow_count"] + global_mean_cat * alpha_cat) / (route_dow_stats_cat["route_dow_count"] + alpha_cat)

route_hour_dow_stats_cat = fit_df_cat.groupby(["route_id", "hour_slot", "dayofweek"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_cat = route_hour_dow_stats_cat.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_cat["target_smooth_route_hour_dow"] = (route_hour_dow_stats_cat["route_hour_dow_mean"] * route_hour_dow_stats_cat["route_hour_dow_count"] + global_mean_cat * alpha_cat) / (route_hour_dow_stats_cat["route_hour_dow_count"] + alpha_cat)


In [ ]:
# Добавляем сглаженные признаки в train и validation.
fit_df_cat_features = fit_df_cat.merge(route_stats_cat[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
fit_df_cat_features = fit_df_cat_features.merge(route_hour_stats_cat[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
fit_df_cat_features = fit_df_cat_features.merge(route_dow_stats_cat[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
fit_df_cat_features = fit_df_cat_features.merge(route_hour_dow_stats_cat[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

val_df_cat_features = val_df_cat.merge(route_stats_cat[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
val_df_cat_features = val_df_cat_features.merge(route_hour_stats_cat[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
val_df_cat_features = val_df_cat_features.merge(route_dow_stats_cat[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
val_df_cat_features = val_df_cat_features.merge(route_hour_dow_stats_cat[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

smooth_features_cat = ["target_smooth_route", "target_smooth_route_hour", "target_smooth_route_dow", "target_smooth_route_hour_dow"]

for frame in (fit_df_cat_features, val_df_cat_features):
    frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean_cat)
    frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])


In [ ]:
# Обучаем CatBoost и сравниваем его с текущим локальным ориентиром.
feature_cols_cat = base_features_cat + smooth_features_cat
cat_features_idx = [feature_cols_cat.index('route_id')]

X_fit_cat = fit_df_cat_features[feature_cols_cat].copy()
X_val_cat = val_df_cat_features[feature_cols_cat].copy()
y_fit_cat = fit_df_cat_features[target_cat_col].to_numpy()
y_val_cat = val_df_cat_features[target_cat_col].to_numpy()

cat_model = CatBoostRegressor(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=100,
)

cat_model.fit(
    X_fit_cat,
    y_fit_cat,
    eval_set=(X_val_cat, y_val_cat),
    cat_features=cat_features_idx,
    use_best_model=True,
    early_stopping_rounds=100,
)

pred_cat_local = cat_model.predict(X_val_cat)
cat_metrics = competition_metric(y_val_cat, pred_cat_local)

print("Текущий лучший public ориентир: 0.3608341250041473")
print("Текущий лучший локальный ориентир smoothing-версии: 0.3835836125379652")
print("Локальный score CatBoost-версии:", cat_metrics["score"])
print(json.dumps(cat_metrics, indent=2, ensure_ascii=False))


Как интерпретировать результат `CatBoost`:

- если локальный `score` меньше `0.3835836125379652`, CatBoost выглядит перспективно
- если локальный `score` примерно равен лучшему LightGBM, можно потом попробовать blend
- если локальный `score` хуже, CatBoost в таком виде не даёт прироста


Финальный сабмит для версии с `CatBoost`.

Этот блок запускайте только если локально CatBoost выглядит не хуже текущей лучшей версии.


In [ ]:
# Строим те же признаки по всему train и test для финального CatBoost-сабмита.
global_mean_cat_full = float(train_df_cat[target_cat_col].mean())

route_stats_cat_full = train_df_cat.groupby(["route_id"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_stats_cat_full = route_stats_cat_full.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_cat_full["target_smooth_route"] = (route_stats_cat_full["route_mean"] * route_stats_cat_full["route_count"] + global_mean_cat_full * alpha_cat) / (route_stats_cat_full["route_count"] + alpha_cat)

route_hour_stats_cat_full = train_df_cat.groupby(["route_id", "hour_slot"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_hour_stats_cat_full = route_hour_stats_cat_full.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_cat_full["target_smooth_route_hour"] = (route_hour_stats_cat_full["route_hour_mean"] * route_hour_stats_cat_full["route_hour_count"] + global_mean_cat_full * alpha_cat) / (route_hour_stats_cat_full["route_hour_count"] + alpha_cat)

route_dow_stats_cat_full = train_df_cat.groupby(["route_id", "dayofweek"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_dow_stats_cat_full = route_dow_stats_cat_full.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_cat_full["target_smooth_route_dow"] = (route_dow_stats_cat_full["route_dow_mean"] * route_dow_stats_cat_full["route_dow_count"] + global_mean_cat_full * alpha_cat) / (route_dow_stats_cat_full["route_dow_count"] + alpha_cat)

route_hour_dow_stats_cat_full = train_df_cat.groupby(["route_id", "hour_slot", "dayofweek"])[target_cat_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_cat_full = route_hour_dow_stats_cat_full.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_cat_full["target_smooth_route_hour_dow"] = (route_hour_dow_stats_cat_full["route_hour_dow_mean"] * route_hour_dow_stats_cat_full["route_hour_dow_count"] + global_mean_cat_full * alpha_cat) / (route_hour_dow_stats_cat_full["route_hour_dow_count"] + alpha_cat)


In [ ]:
# Добавляем признаки и сохраняем отдельный сабмит для CatBoost.
train_full_cat = train_df_cat.merge(route_stats_cat_full[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
train_full_cat = train_full_cat.merge(route_hour_stats_cat_full[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
train_full_cat = train_full_cat.merge(route_dow_stats_cat_full[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
train_full_cat = train_full_cat.merge(route_hour_dow_stats_cat_full[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

test_full_cat = test_df_cat.merge(route_stats_cat_full[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
test_full_cat = test_full_cat.merge(route_hour_stats_cat_full[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
test_full_cat = test_full_cat.merge(route_dow_stats_cat_full[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
test_full_cat = test_full_cat.merge(route_hour_dow_stats_cat_full[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

for frame in (train_full_cat, test_full_cat):
    frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean_cat_full)
    frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])

X_train_cat_full = train_full_cat[feature_cols_cat].copy()
X_test_cat_full = test_full_cat[feature_cols_cat].copy()
y_train_cat_full = train_full_cat[target_cat_col].to_numpy()

final_cat_model = CatBoostRegressor(
    loss_function='MAE',
    iterations=int(cat_model.get_best_iteration() or 1000),
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=False,
)

final_cat_model.fit(
    X_train_cat_full,
    y_train_cat_full,
    cat_features=cat_features_idx,
)

pred_test_cat = np.clip(final_cat_model.predict(X_test_cat_full), a_min=0, a_max=None)
submission_cat = pd.DataFrame({
    "id": test_df_cat["id"],
    "y_pred": pred_test_cat,
})

submission_cat_path = "submission_route_time_agg_catboost_v1.csv"
submission_cat.to_csv(submission_cat_path, index=False)

print(f"Сабмит CatBoost сохранён в файл: {submission_cat_path}")
submission_cat.head()


Заметка для презентации по версии с `CatBoost`:

- Что проверяли: другую модель на тех же сильных признаках
- База признаков: сглаженные агрегаты с `alpha = 20`
- Локальный score: `0.38463582160697896`
- Локальный ориентир лучшей LightGBM-версии: `0.3835836125379652`
- Вывод: CatBoost близко, но всё же хуже как отдельная модель; пробуем blend


Следующий эксперимент: blend `LightGBM + CatBoost`.

Идея: модели близки по качеству, но ошибаются по-разному.
Даже если CatBoost чуть хуже отдельно, в смеси он может улучшить итоговый score.


In [ ]:
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

from forecast_pipeline import (
    competition_metric,
    fit_and_predict,
    make_time_features,
    make_validation_split,
    train_full_and_predict_test,
)


In [ ]:
# Загружаем train и test заново, чтобы блок был полностью самостоятельным.
train_df_blend = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_blend = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_blend = make_time_features(train_df_blend)
test_df_blend = make_time_features(test_df_blend)

fit_df_blend, val_df_blend = make_validation_split(train_df_blend, val_horizon=8)

base_features_blend = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_blend_col = "target_1h"
alpha_blend = 20.0


In [ ]:
# Строим лучшие на сегодня сглаженные признаки для обеих моделей.
global_mean_blend = float(fit_df_blend[target_blend_col].mean())

route_stats_blend = fit_df_blend.groupby(["route_id"])[target_blend_col].agg(["mean", "size"]).reset_index()
route_stats_blend = route_stats_blend.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_blend["target_smooth_route"] = (route_stats_blend["route_mean"] * route_stats_blend["route_count"] + global_mean_blend * alpha_blend) / (route_stats_blend["route_count"] + alpha_blend)

route_hour_stats_blend = fit_df_blend.groupby(["route_id", "hour_slot"])[target_blend_col].agg(["mean", "size"]).reset_index()
route_hour_stats_blend = route_hour_stats_blend.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_blend["target_smooth_route_hour"] = (route_hour_stats_blend["route_hour_mean"] * route_hour_stats_blend["route_hour_count"] + global_mean_blend * alpha_blend) / (route_hour_stats_blend["route_hour_count"] + alpha_blend)

route_dow_stats_blend = fit_df_blend.groupby(["route_id", "dayofweek"])[target_blend_col].agg(["mean", "size"]).reset_index()
route_dow_stats_blend = route_dow_stats_blend.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_blend["target_smooth_route_dow"] = (route_dow_stats_blend["route_dow_mean"] * route_dow_stats_blend["route_dow_count"] + global_mean_blend * alpha_blend) / (route_dow_stats_blend["route_dow_count"] + alpha_blend)

route_hour_dow_stats_blend = fit_df_blend.groupby(["route_id", "hour_slot", "dayofweek"])[target_blend_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_blend = route_hour_dow_stats_blend.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_blend["target_smooth_route_hour_dow"] = (route_hour_dow_stats_blend["route_hour_dow_mean"] * route_hour_dow_stats_blend["route_hour_dow_count"] + global_mean_blend * alpha_blend) / (route_hour_dow_stats_blend["route_hour_dow_count"] + alpha_blend)


In [ ]:
# Добавляем сглаженные признаки в train и validation.
fit_df_blend_features = fit_df_blend.merge(route_stats_blend[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
fit_df_blend_features = fit_df_blend_features.merge(route_hour_stats_blend[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
fit_df_blend_features = fit_df_blend_features.merge(route_dow_stats_blend[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
fit_df_blend_features = fit_df_blend_features.merge(route_hour_dow_stats_blend[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

val_df_blend_features = val_df_blend.merge(route_stats_blend[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
val_df_blend_features = val_df_blend_features.merge(route_hour_stats_blend[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
val_df_blend_features = val_df_blend_features.merge(route_dow_stats_blend[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
val_df_blend_features = val_df_blend_features.merge(route_hour_dow_stats_blend[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")

smooth_features_blend = ["target_smooth_route", "target_smooth_route_hour", "target_smooth_route_dow", "target_smooth_route_hour_dow"]

for frame in (fit_df_blend_features, val_df_blend_features):
    frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean_blend)
    frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])


In [ ]:
# Обучаем LightGBM на тех же признаках.
_, pred_lgb_local = fit_and_predict(
    fit_df_blend_features,
    val_df_blend_features,
    feature_cols=base_features_blend + smooth_features_blend,
)

# Обучаем CatBoost на тех же признаках.
feature_cols_blend = base_features_blend + smooth_features_blend
cat_features_idx_blend = [feature_cols_blend.index('route_id')]
X_fit_blend = fit_df_blend_features[feature_cols_blend].copy()
X_val_blend = val_df_blend_features[feature_cols_blend].copy()
y_fit_blend = fit_df_blend_features[target_blend_col].to_numpy()
y_val_blend = val_df_blend_features[target_blend_col].to_numpy()

cat_model_blend = CatBoostRegressor(
    loss_function='MAE',
    eval_metric='MAE',
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=100,
)

cat_model_blend.fit(
    X_fit_blend,
    y_fit_blend,
    eval_set=(X_val_blend, y_val_blend),
    cat_features=cat_features_idx_blend,
    use_best_model=True,
    early_stopping_rounds=100,
)

pred_cat_local = cat_model_blend.predict(X_val_blend)


In [ ]:
# Проверяем несколько весов blend между LightGBM и CatBoost.
blend_weights = [0.5, 0.6, 0.7, 0.8, 0.9]
blend_results = []

for lgb_weight in blend_weights:
    cat_weight = 1.0 - lgb_weight
    pred_blend = lgb_weight * pred_lgb_local + cat_weight * pred_cat_local
    metrics_blend = competition_metric(y_val_blend, pred_blend)
    blend_results.append({
        "lgb_weight": lgb_weight,
        "cat_weight": cat_weight,
        "wape": metrics_blend["wape"],
        "relative_bias": metrics_blend["relative_bias"],
        "score": metrics_blend["score"],
    })

blend_results_df = pd.DataFrame(blend_results).sort_values("score")
blend_results_df


In [ ]:
# Показываем лучший blend и сравниваем его с текущим лучшим ориентиром.
best_blend_row = blend_results_df.iloc[0]
print("Локальный ориентир лучшей LightGBM-версии:", 0.3835836125379652)
print("Локальный score CatBoost:", 0.38463582160697896)
print("Лучший вес LightGBM:", best_blend_row["lgb_weight"])
print("Лучший вес CatBoost:", best_blend_row["cat_weight"])
print("Лучший локальный score blend:", best_blend_row["score"])
print("Полная таблица blend-результатов:")
print(blend_results_df.to_string(index=False))


Как интерпретировать результат blend:

- если лучший blend-score меньше `0.3835836125379652`, blend стоит проверить на leaderboard
- если blend хуже LightGBM, оставляем текущую лучшую smoothing-версию
- если blend чуть лучше локально, его всё равно стоит проверить, потому что на public leaderboard смесь может повести себя лучше


Заметка для презентации по blend-эксперименту:

- Что делали: смешивали предсказания LightGBM и CatBoost
- База признаков: сглаженные агрегаты с `alpha = 20`
- Проверяемые веса LightGBM: `0.5, 0.6, 0.7, 0.8, 0.9`
- Лучший вес: `вписать после запуска`
- Лучший локальный score blend: `вписать после запуска`
- Вывод: `вписать после сравнения с лучшей LightGBM-версией`


Итог по финальному blend-сабмиту:

- Локально лучший blend:
  - `LightGBM = 0.8`
  - `CatBoost = 0.2`
  - локальный score: `0.3834913913957258`
- Public leaderboard:
  - `0.36264498745756096`
- Вывод:
  - blend оказался хуже лучшей smoothing-версии с `0.3608341250041473`
  - как боевую основную версию blend не берём


Итог по эксперименту с более точными временными агрегатами:

- Что пробовали:
  - `target_smooth_route_hour_day`
  - `target_smooth_route_hour_month`
- Локальный score:
  - `0.46708208227641973`
- Вывод:
  - комбинации оказались слишком шумными
  - сильно выросли и `WAPE`, и `relative_bias`
  - эту ветку закрываем, сабмит не делаем


Сводка перед последним экспериментом:

- Лучший текущий public result:
  - `0.3608341250041473`
  - файл: `submission_route_time_agg_smooth_v1.csv`
- Неудачные ветки:
  - `count`
  - `median`
  - `blend`
  - слишком точные временные агрегаты
- Текущий последний безопасный шаг:
  - более устойчивые агрегаты по `month`, `is_weekend` и `dayofweek + month`


Последний безопасный эксперимент: более устойчивые агрегаты.

После неудачи с слишком локальными временными признаками пробуем более стабильные исторические паттерны:

- сглаженное среднее по `(route_id, month)`
- сглаженное среднее по `(route_id, is_weekend)`
- сглаженное среднее по `(route_id, dayofweek, month)`

Идея: дать модели дополнительную сезонность, но не уходить в слишком редкие комбинации.


In [ ]:
import json
import numpy as np
import pandas as pd

from forecast_pipeline import (
    competition_metric,
    fit_and_predict,
    make_time_features,
    make_validation_split,
    train_full_and_predict_test,
)


In [ ]:
# Загружаем train и test заново, чтобы блок был полностью самостоятельным.
train_df_stable = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_stable = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_stable = make_time_features(train_df_stable)
test_df_stable = make_time_features(test_df_stable)

fit_df_stable, val_df_stable = make_validation_split(train_df_stable, val_horizon=8)

base_features_stable = [
    "route_id",
    "hour",
    "minute",
    "hour_slot",
    "dayofweek",
    "day",
    "month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
]

target_stable_col = "target_1h"
alpha_stable = 20.0
known_smoothing_score = 0.3835836125379652


In [ ]:
# Строим текущие лучшие сглаженные признаки.
global_mean_stable = float(fit_df_stable[target_stable_col].mean())

route_stats_stable = fit_df_stable.groupby(["route_id"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_stats_stable = route_stats_stable.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_stable["target_smooth_route"] = (route_stats_stable["route_mean"] * route_stats_stable["route_count"] + global_mean_stable * alpha_stable) / (route_stats_stable["route_count"] + alpha_stable)

route_hour_stats_stable = fit_df_stable.groupby(["route_id", "hour_slot"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_hour_stats_stable = route_hour_stats_stable.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_stable["target_smooth_route_hour"] = (route_hour_stats_stable["route_hour_mean"] * route_hour_stats_stable["route_hour_count"] + global_mean_stable * alpha_stable) / (route_hour_stats_stable["route_hour_count"] + alpha_stable)

route_dow_stats_stable = fit_df_stable.groupby(["route_id", "dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_dow_stats_stable = route_dow_stats_stable.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_stable["target_smooth_route_dow"] = (route_dow_stats_stable["route_dow_mean"] * route_dow_stats_stable["route_dow_count"] + global_mean_stable * alpha_stable) / (route_dow_stats_stable["route_dow_count"] + alpha_stable)

route_hour_dow_stats_stable = fit_df_stable.groupby(["route_id", "hour_slot", "dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_stable = route_hour_dow_stats_stable.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_stable["target_smooth_route_hour_dow"] = (route_hour_dow_stats_stable["route_hour_dow_mean"] * route_hour_dow_stats_stable["route_hour_dow_count"] + global_mean_stable * alpha_stable) / (route_hour_dow_stats_stable["route_hour_dow_count"] + alpha_stable)

global_hour_stats_stable = fit_df_stable.groupby(["hour_slot"])[target_stable_col].agg(["mean", "size"]).reset_index()
global_hour_stats_stable = global_hour_stats_stable.rename(columns={"mean": "global_hour_mean", "size": "global_hour_count"})
global_hour_stats_stable["target_smooth_global_hour"] = (global_hour_stats_stable["global_hour_mean"] * global_hour_stats_stable["global_hour_count"] + global_mean_stable * alpha_stable) / (global_hour_stats_stable["global_hour_count"] + alpha_stable)

global_dow_stats_stable = fit_df_stable.groupby(["dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
global_dow_stats_stable = global_dow_stats_stable.rename(columns={"mean": "global_dow_mean", "size": "global_dow_count"})
global_dow_stats_stable["target_smooth_global_dow"] = (global_dow_stats_stable["global_dow_mean"] * global_dow_stats_stable["global_dow_count"] + global_mean_stable * alpha_stable) / (global_dow_stats_stable["global_dow_count"] + alpha_stable)

smooth_features_stable = [
    "target_smooth_route",
    "target_smooth_route_hour",
    "target_smooth_route_dow",
    "target_smooth_route_hour_dow",
    "target_smooth_global_hour",
    "target_smooth_global_dow",
]


In [ ]:
# Строим новые более устойчивые агрегаты.
route_month_stats = fit_df_stable.groupby(["route_id", "month"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_month_stats = route_month_stats.rename(columns={"mean": "route_month_mean", "size": "route_month_count"})
route_month_stats["target_smooth_route_month"] = (route_month_stats["route_month_mean"] * route_month_stats["route_month_count"] + global_mean_stable * alpha_stable) / (route_month_stats["route_month_count"] + alpha_stable)

route_weekend_stats = fit_df_stable.groupby(["route_id", "is_weekend"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_weekend_stats = route_weekend_stats.rename(columns={"mean": "route_weekend_mean", "size": "route_weekend_count"})
route_weekend_stats["target_smooth_route_weekend"] = (route_weekend_stats["route_weekend_mean"] * route_weekend_stats["route_weekend_count"] + global_mean_stable * alpha_stable) / (route_weekend_stats["route_weekend_count"] + alpha_stable)

route_dow_month_stats = fit_df_stable.groupby(["route_id", "dayofweek", "month"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_dow_month_stats = route_dow_month_stats.rename(columns={"mean": "route_dow_month_mean", "size": "route_dow_month_count"})
route_dow_month_stats["target_smooth_route_dow_month"] = (route_dow_month_stats["route_dow_month_mean"] * route_dow_month_stats["route_dow_month_count"] + global_mean_stable * alpha_stable) / (route_dow_month_stats["route_dow_month_count"] + alpha_stable)

stable_extra_features = [
    "target_smooth_route_month",
    "target_smooth_route_weekend",
    "target_smooth_route_dow_month",
]


In [ ]:
# Добавляем признаки в fit/validation и ужимаем типы, чтобы не перегружать память.
fit_df_stable_features = fit_df_stable.copy()
val_df_stable_features = val_df_stable.copy()

for frame_name in ("fit_df_stable_features", "val_df_stable_features"):
    frame = locals()[frame_name]
    frame = frame.merge(route_stats_stable[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
    frame = frame.merge(route_hour_stats_stable[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
    frame = frame.merge(route_dow_stats_stable[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
    frame = frame.merge(route_hour_dow_stats_stable[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")
    frame = frame.merge(global_hour_stats_stable[["hour_slot", "target_smooth_global_hour"]], on=["hour_slot"], how="left")
    frame = frame.merge(global_dow_stats_stable[["dayofweek", "target_smooth_global_dow"]], on=["dayofweek"], how="left")
    frame = frame.merge(route_month_stats[["route_id", "month", "target_smooth_route_month"]], on=["route_id", "month"], how="left")
    frame = frame.merge(route_weekend_stats[["route_id", "is_weekend", "target_smooth_route_weekend"]], on=["route_id", "is_weekend"], how="left")
    frame = frame.merge(route_dow_month_stats[["route_id", "dayofweek", "month", "target_smooth_route_dow_month"]], on=["route_id", "dayofweek", "month"], how="left")

    frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean_stable)
    frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])
    frame["target_smooth_global_hour"] = frame["target_smooth_global_hour"].fillna(global_mean_stable)
    frame["target_smooth_global_dow"] = frame["target_smooth_global_dow"].fillna(global_mean_stable)
    frame["target_smooth_route_month"] = frame["target_smooth_route_month"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_weekend"] = frame["target_smooth_route_weekend"].fillna(frame["target_smooth_route_dow"])
    frame["target_smooth_route_dow_month"] = frame["target_smooth_route_dow_month"].fillna(frame["target_smooth_route_dow"])

    for col in smooth_features_stable + stable_extra_features:
        frame[col] = frame[col].astype("float32")

    locals()[frame_name] = frame


In [ ]:
# Сравниваем новый вариант с уже известным лучшим локальным ориентиром.
_, pred_stable_local = fit_and_predict(
    fit_df_stable_features,
    val_df_stable_features,
    feature_cols=base_features_stable + smooth_features_stable + stable_extra_features,
)

y_val_stable = val_df_stable[target_stable_col].to_numpy()
stable_metrics_local = competition_metric(y_val_stable, pred_stable_local)

print("Известный локальный score лучшей smoothing-версии:", known_smoothing_score)
print("Локальный score версии с более устойчивыми агрегатами:", stable_metrics_local["score"])
print()
print("Метрики версии с более устойчивыми агрегатами:")
print(json.dumps(stable_metrics_local, indent=2, ensure_ascii=False))


Как интерпретировать результат эксперимента с более устойчивыми агрегатами:

- если новый score меньше `0.3835836125379652`, идею стоит проверить на leaderboard
- если новая версия хуже, остаёмся на текущей лучшей smoothing-версии
- это последний безопасный эксперимент: он добавляет сезонность, но не уходит в редкие комбинации


In [ ]:
# Если локально стало лучше, обучаем финальную версию на всём train и формируем submission.
train_df_stable_full = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\train_solo_track.parquet")
test_df_stable_full = pd.read_parquet(r"D:\ml\WB-hackathon\vscode\test_solo_track.parquet")

train_df_stable_full = make_time_features(train_df_stable_full)
test_df_stable_full = make_time_features(test_df_stable_full)

global_mean_stable_full = float(train_df_stable_full[target_stable_col].mean())

route_stats_stable_full = train_df_stable_full.groupby(["route_id"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_stats_stable_full = route_stats_stable_full.rename(columns={"mean": "route_mean", "size": "route_count"})
route_stats_stable_full["target_smooth_route"] = (route_stats_stable_full["route_mean"] * route_stats_stable_full["route_count"] + global_mean_stable_full * alpha_stable) / (route_stats_stable_full["route_count"] + alpha_stable)

route_hour_stats_stable_full = train_df_stable_full.groupby(["route_id", "hour_slot"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_hour_stats_stable_full = route_hour_stats_stable_full.rename(columns={"mean": "route_hour_mean", "size": "route_hour_count"})
route_hour_stats_stable_full["target_smooth_route_hour"] = (route_hour_stats_stable_full["route_hour_mean"] * route_hour_stats_stable_full["route_hour_count"] + global_mean_stable_full * alpha_stable) / (route_hour_stats_stable_full["route_hour_count"] + alpha_stable)

route_dow_stats_stable_full = train_df_stable_full.groupby(["route_id", "dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_dow_stats_stable_full = route_dow_stats_stable_full.rename(columns={"mean": "route_dow_mean", "size": "route_dow_count"})
route_dow_stats_stable_full["target_smooth_route_dow"] = (route_dow_stats_stable_full["route_dow_mean"] * route_dow_stats_stable_full["route_dow_count"] + global_mean_stable_full * alpha_stable) / (route_dow_stats_stable_full["route_dow_count"] + alpha_stable)

route_hour_dow_stats_stable_full = train_df_stable_full.groupby(["route_id", "hour_slot", "dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_hour_dow_stats_stable_full = route_hour_dow_stats_stable_full.rename(columns={"mean": "route_hour_dow_mean", "size": "route_hour_dow_count"})
route_hour_dow_stats_stable_full["target_smooth_route_hour_dow"] = (route_hour_dow_stats_stable_full["route_hour_dow_mean"] * route_hour_dow_stats_stable_full["route_hour_dow_count"] + global_mean_stable_full * alpha_stable) / (route_hour_dow_stats_stable_full["route_hour_dow_count"] + alpha_stable)

global_hour_stats_stable_full = train_df_stable_full.groupby(["hour_slot"])[target_stable_col].agg(["mean", "size"]).reset_index()
global_hour_stats_stable_full = global_hour_stats_stable_full.rename(columns={"mean": "global_hour_mean", "size": "global_hour_count"})
global_hour_stats_stable_full["target_smooth_global_hour"] = (global_hour_stats_stable_full["global_hour_mean"] * global_hour_stats_stable_full["global_hour_count"] + global_mean_stable_full * alpha_stable) / (global_hour_stats_stable_full["global_hour_count"] + alpha_stable)

global_dow_stats_stable_full = train_df_stable_full.groupby(["dayofweek"])[target_stable_col].agg(["mean", "size"]).reset_index()
global_dow_stats_stable_full = global_dow_stats_stable_full.rename(columns={"mean": "global_dow_mean", "size": "global_dow_count"})
global_dow_stats_stable_full["target_smooth_global_dow"] = (global_dow_stats_stable_full["global_dow_mean"] * global_dow_stats_stable_full["global_dow_count"] + global_mean_stable_full * alpha_stable) / (global_dow_stats_stable_full["global_dow_count"] + alpha_stable)

route_month_stats_full = train_df_stable_full.groupby(["route_id", "month"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_month_stats_full = route_month_stats_full.rename(columns={"mean": "route_month_mean", "size": "route_month_count"})
route_month_stats_full["target_smooth_route_month"] = (route_month_stats_full["route_month_mean"] * route_month_stats_full["route_month_count"] + global_mean_stable_full * alpha_stable) / (route_month_stats_full["route_month_count"] + alpha_stable)

route_weekend_stats_full = train_df_stable_full.groupby(["route_id", "is_weekend"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_weekend_stats_full = route_weekend_stats_full.rename(columns={"mean": "route_weekend_mean", "size": "route_weekend_count"})
route_weekend_stats_full["target_smooth_route_weekend"] = (route_weekend_stats_full["route_weekend_mean"] * route_weekend_stats_full["route_weekend_count"] + global_mean_stable_full * alpha_stable) / (route_weekend_stats_full["route_weekend_count"] + alpha_stable)

route_dow_month_stats_full = train_df_stable_full.groupby(["route_id", "dayofweek", "month"])[target_stable_col].agg(["mean", "size"]).reset_index()
route_dow_month_stats_full = route_dow_month_stats_full.rename(columns={"mean": "route_dow_month_mean", "size": "route_dow_month_count"})
route_dow_month_stats_full["target_smooth_route_dow_month"] = (route_dow_month_stats_full["route_dow_month_mean"] * route_dow_month_stats_full["route_dow_month_count"] + global_mean_stable_full * alpha_stable) / (route_dow_month_stats_full["route_dow_month_count"] + alpha_stable)

train_full_stable_features = train_df_stable_full.copy()
test_full_stable_features = test_df_stable_full.copy()

for frame_name in ("train_full_stable_features", "test_full_stable_features"):
    frame = locals()[frame_name]
    frame = frame.merge(route_stats_stable_full[["route_id", "target_smooth_route"]], on=["route_id"], how="left")
    frame = frame.merge(route_hour_stats_stable_full[["route_id", "hour_slot", "target_smooth_route_hour"]], on=["route_id", "hour_slot"], how="left")
    frame = frame.merge(route_dow_stats_stable_full[["route_id", "dayofweek", "target_smooth_route_dow"]], on=["route_id", "dayofweek"], how="left")
    frame = frame.merge(route_hour_dow_stats_stable_full[["route_id", "hour_slot", "dayofweek", "target_smooth_route_hour_dow"]], on=["route_id", "hour_slot", "dayofweek"], how="left")
    frame = frame.merge(global_hour_stats_stable_full[["hour_slot", "target_smooth_global_hour"]], on=["hour_slot"], how="left")
    frame = frame.merge(global_dow_stats_stable_full[["dayofweek", "target_smooth_global_dow"]], on=["dayofweek"], how="left")
    frame = frame.merge(route_month_stats_full[["route_id", "month", "target_smooth_route_month"]], on=["route_id", "month"], how="left")
    frame = frame.merge(route_weekend_stats_full[["route_id", "is_weekend", "target_smooth_route_weekend"]], on=["route_id", "is_weekend"], how="left")
    frame = frame.merge(route_dow_month_stats_full[["route_id", "dayofweek", "month", "target_smooth_route_dow_month"]], on=["route_id", "dayofweek", "month"], how="left")
    frame["target_smooth_route"] = frame["target_smooth_route"].fillna(global_mean_stable_full)
    frame["target_smooth_route_hour"] = frame["target_smooth_route_hour"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_dow"] = frame["target_smooth_route_dow"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_hour_dow"] = frame["target_smooth_route_hour_dow"].fillna(frame["target_smooth_route_hour"])
    frame["target_smooth_global_hour"] = frame["target_smooth_global_hour"].fillna(global_mean_stable_full)
    frame["target_smooth_global_dow"] = frame["target_smooth_global_dow"].fillna(global_mean_stable_full)
    frame["target_smooth_route_month"] = frame["target_smooth_route_month"].fillna(frame["target_smooth_route"])
    frame["target_smooth_route_weekend"] = frame["target_smooth_route_weekend"].fillna(frame["target_smooth_route_dow"])
    frame["target_smooth_route_dow_month"] = frame["target_smooth_route_dow_month"].fillna(frame["target_smooth_route_dow"])
    locals()[frame_name] = frame

_, pred_stable_test = train_full_and_predict_test(
    train_full_stable_features,
    test_full_stable_features,
    feature_cols=base_features_stable + smooth_features_stable + stable_extra_features,
)

pred_stable_test = np.clip(pred_stable_test, a_min=0, a_max=None)


In [ ]:
# Формируем submission для версии с более устойчивыми агрегатами.
submission_stable = pd.DataFrame({
    "id": test_full_stable_features["id"],
    "y_pred": pred_stable_test,
})

submission_stable.to_csv("submission_route_time_agg_stable_v1.csv", index=False)
print("Файл сохранён: submission_route_time_agg_stable_v1.csv")
submission_stable.head()


Заметка для презентации по эксперименту с более устойчивыми агрегатами:

- Что добавили:
  - `target_smooth_route_month`
  - `target_smooth_route_weekend`
  - `target_smooth_route_dow_month`
- Зачем:
  - добавить сезонность без слишком редких комбинаций
- Локальный score: `вписать после запуска`
- Public leaderboard score: `вписать после загрузки`
- Вывод: `вписать после сравнения с лучшей smoothing-версией`
